# Mision 2: Buscando Patrones


**Hoy vas a trabajar con un dataset real:** el World Happiness Report 2019 --
puntaje de felicidad de 156 paises, junto con seis variables economicas y
sociales que podrian estar relacionadas con ese puntaje. La Mision 1 te enseño
a resumir una sola columna con honestidad. Esta mision te enseña algo nuevo:
**como saber si dos columnas se mueven juntas** -- y que tan peligroso es leer
esa relacion como algo mas de lo que realmente es.

---

### Leyenda de iconos

| Icono | Accion | Que significa |
|---|---|---|
| 👀 | **OBSERVA** | Ejecuta y observa -- no cambies nada |
| ✏️ | **MODIFICA** | Edita y vuelve a ejecutar |
| 🔮 | **PREDICE** | Escribe tu prediccion *antes* de ejecutar |
| 🧩 | **COMPLETA** | Reemplaza `___` por el valor correcto |
| 🔨 | **CONSTRUYE** | Escribe codigo desde cero en el bloque "Tu codigo aqui" |
| 🔧 | **DEBUG** | Ejecuta, lee el error, corrigelo |
| ✅ | **VERIFICA** | El autograder revisa tu respuesta |
| ❓ | **TEORIA** | Pregunta de opcion multiple (se muestra al ejecutar la celda) |
| 💭 | **REFLEXIONA** | Respuesta abierta -- calificada por IA, feedback instantaneo (+5 XP) |

---


In [ ]:
# Carga el autograder y el dataset de esta leccion
!wget -q "https://raw.githubusercontent.com/Santa-Maria-de-los-Andes/intro_to_stats/main/course-python-stats/Weeks%203-4/autograder_nb3.py"
!wget -q "https://raw.githubusercontent.com/Santa-Maria-de-los-Andes/intro_to_stats/main/course-python-stats/Weeks%203-4/2019_es.csv"
from autograder_nb3 import Autograder
grader = Autograder()

import pandas as pd

df_felicidad = pd.read_csv('2019_es.csv')

print("Dataset cargado.")

## 🌍 Un vistazo rapido antes de empezar

`df_felicidad` tiene **156 filas** (un pais o region por fila) y **10
columnas**: el puesto en el ranking, el nombre del pais, el continente, el
`Puntaje` de felicidad, y seis variables que el reporte usa para explicar ese
puntaje (`PBI per cápita`, `Apoyo social`, `Esperanza de vida saludable`,
`Libertad para tomar decisiones`, `Generosidad`, `Percepción de corrupción`).

A diferencia del dataset de videojuegos de la Mision 1, **este no tiene
ningun valor faltante** -- las 156 filas estan completas en las 10 columnas.
Eso no significa que el dataset sea perfecto (mas adelante vas a ver que una
de sus columnas es, en realidad, un espejo de otra) -- solo significa que hoy
el problema no es "que falta," sino **"que tan honesto es el patron que
creo ver."**


## 🎬 Apertura -- Tres Diagramas


Abajo hay **tres diagramas de dispersion** reales, calculados con datos reales
de `df_felicidad`. Los ejes dicen solo "Variable X" y "Variable Y" a
proposito -- todavia no sabes que columnas son.

**Tu tarea antes de seguir:** ordena los tres diagramas de **mas
patron** (los puntos forman una linea reconocible) a **menos patron** (una
nube sin forma clara). No hay codigo que calcular todavia -- solo tu ojo.


In [ ]:
# 👀 OBSERVA: tres diagramas reales, ejes sin identificar a proposito
import matplotlib.pyplot as plt

pares_apertura = [
    ('Libertad para tomar decisiones', 'Puntaje'),   # Diagrama 1
    ('Generosidad', 'Puntaje'),                       # Diagrama 2
    ('PBI per cápita', 'Puntaje'),                    # Diagrama 3
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for i, (ax, (col_x, col_y)) in enumerate(zip(axes, pares_apertura), start=1):
    ax.scatter(df_felicidad[col_x], df_felicidad[col_y], alpha=0.6)
    ax.set_xlabel('Variable X')
    ax.set_ylabel('Variable Y')
    ax.set_title(f'Diagrama {i}')
plt.tight_layout()
plt.show()

#### Antes de seguir -- tu orden

Escribe tu orden de "mas patron" a "menos patron" (ej. `"2, 3, 1"`).

In [ ]:
# 🔮 PREDICE (no se califica, es solo para ti)
mi_orden_apertura = "___"  # ej: "Diagrama 3 es el mas fuerte, Diagrama 2 el mas debil"

In [ ]:
# 👀 OBSERVA: la revelacion -- mismos tres diagramas, ahora con nombres reales y su r
# El patron que vas a repetir todo el dia es siempre este:
#     df_felicidad['columna_x'].corr(df_felicidad['columna_y'])
# Aqui lo aplicamos tres veces, una por diagrama, con las columnas ya identificadas.

r_diagrama1 = df_felicidad['Libertad para tomar decisiones'].corr(df_felicidad['Puntaje'])
r_diagrama2 = df_felicidad['Generosidad'].corr(df_felicidad['Puntaje'])
r_diagrama3 = df_felicidad['PBI per cápita'].corr(df_felicidad['Puntaje'])

print("Diagrama 1 (Libertad para tomar decisiones vs. Puntaje) -> r =", round(r_diagrama1, 3))
print("Diagrama 2 (Generosidad vs. Puntaje)                    -> r =", round(r_diagrama2, 3))
print("Diagrama 3 (PBI per cápita vs. Puntaje)                 -> r =", round(r_diagrama3, 3))

**Diagrama 3 (PBI per cápita) es el patron mas fuerte** (r ≈ 0.79): los
paises con mayor produccion economica por persona tienden a reportar mayor
puntaje de felicidad. **Diagrama 2 (Generosidad) es casi una nube sin forma**
(r ≈ 0.08): saber cuanto dona en promedio la gente de un pais casi no te dice
nada sobre su puntaje de felicidad. Diagrama 1 (Libertad para tomar
decisiones) queda en medio (r ≈ 0.57) -- un patron real, pero mucho menos
limpio que el del PBI.

> ⚠️ Ningun numero de esta apertura dice **por que** pasa esto. "Los paises
> con mayor PBI per capita tienden a reportar mayor felicidad" es una
> **relacion observada** -- no es lo mismo que "tener mas dinero *causa*
> felicidad." 



## 🔓 Teoria Desbloqueada -- El Coeficiente de Correlacion

### ¿Que es el coeficiente de correlacion?

Un numero que resume **que tan fuerte** y **en que direccion** dos variables
numericas se mueven juntas. En este curso se calcula con `.corr()`
(correlacion de Pearson)

### El rango: -1 a 1

| Valor de r | Que significa |
|---|---|
| Cercano a **+1** | Relacion fuerte y positiva -- cuando una sube, la otra tiende a subir |
| Cercano a **-1** | Relacion fuerte y negativa -- cuando una sube, la otra tiende a bajar |
| Cercano a **0** | Relacion lineal debil o inexistente |

### Dos preguntas distintas: fuerza vs. direccion

- **Direccion** (el signo, + o −): ¿suben juntas o una sube mientras la otra
  baja?
- **Fuerza** (que tan lejos de 0): ¿que tan consistente es ese patron? Un r
  de 0.9 es un patron mucho mas consistente que uno de 0.3, aunque ambos sean
  positivos.

> ⚠️ **`.corr()` solo mide relacion *lineal*.** Una relacion real y fuerte
> pero curva puede dar un r cercano a 0. Es una frase de honestidad, no una
> unidad nueva -- este curso no enseña a detectar relaciones no lineales.

> ⚠️ **La regla mas importante de estas dos semanas:** un coeficiente de
> correlacion **nunca, por si solo, te dice si una variable causa la otra.**


In [ ]:
# ❓ Pregunta t1 -- ejecuta esta celda para verla y responder
grader.check_t1()

In [ ]:
# ❓ Pregunta t2 -- ejecuta esta celda para verla y responder
grader.check_t2()

In [ ]:
# ❓ Pregunta t3 -- ejecuta esta celda para verla y responder
grader.check_t3()

In [ ]:
# ❓ Pregunta t4 -- ejecuta esta celda para verla y responder
grader.check_t4()

In [ ]:
# ❓ Pregunta t5 -- ejecuta esta celda para verla y responder
grader.check_t5()

---
## 📈🔢 Grafica y Calcula

Hoy graficas y calculas en el mismo paso -- ya sabes leer un patron con el
ojo (Apertura) y ya sabes que significa `r` (Teoria Desbloqueada). El patron
completo que vas a repetir toda la clase:

```python
plt.scatter(df['columna_x'], df['columna_y'])
plt.xlabel('columna_x')
plt.ylabel('columna_y')
plt.show()

r = df['columna_x'].corr(df['columna_y'])
print(f"r = {r:.3f}")
```


In [ ]:
# 👀 OBSERVA: el patron completo, de una vez -- Apoyo social vs. Puntaje
plt.scatter(df_felicidad['Apoyo social'], df_felicidad['Puntaje'], alpha=0.6)
plt.xlabel('Apoyo social')
plt.ylabel('Puntaje')
plt.title('Apoyo social vs. Puntaje de felicidad')
plt.show()

r_apoyo_social = df_felicidad['Apoyo social'].corr(df_felicidad['Puntaje'])
print(f"r (Apoyo social vs. Puntaje) = {r_apoyo_social:.3f}")

### 🔍 ¿Por que el ojo a veces se equivoca?

`Catar` tiene el `PBI per cápita` mas alto de las 156 filas (1.684) -- pero
su `Puntaje` (6.374) **no** es el mas alto del dataset (Finlandia lidera con
7.769). Un solo pais asi ya le resta "limpieza" visual al Diagrama 3 de la
Apertura, que en conjunto sigue siendo fuerte (r ≈ 0.79). Esto es lo normal,
no un error de datos: **ningun par de variables reales forma una linea
perfecta.** Un outlier real puede hacer que tu ojo dude de un patron que el
numero confirma -- o al reves, que tu ojo "vea" un patron que el numero
desmiente.


In [ ]:
# ❓ Pregunta t6 -- ejecuta esta celda para verla y responder
grader.check_t6()

### 🔁 Practica repetida: mismo patron, distintos pares


#### Ronda 1 -- Antes de seguir, predice

Mirando el patron guiado de arriba (Apoyo social vs. Puntaje): ¿te parece un
patron fuerte o debil? ¿positivo o negativo? Ahora, **sin graficar
todavia**, predice lo mismo para un par nuevo: `Percepción de corrupción`
vs. `Puntaje`.


In [ ]:
# 🔮 PREDICE (no se califica, es solo para ti)
mi_prediccion_ronda1 = "___"  # ej: "creo que es un patron debil y positivo"

#### ✅ Ejercicio 1 -- Grafica y calcula (20 pts)

🔨 Repite el patron completo (scatter + `.corr()`) con `Percepción de
corrupción` en el eje X y `Puntaje` en el eje Y. Incluye `plt.xlabel()` e
`plt.ylabel()`.

Variables que espera el autograder: `x_ex1`, `y_ex1` (columnas del scatter),
`r_ex1` (el coeficiente de correlacion entre ambas).


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex1()

#### 💭 Reflexiona -- Ronda 1 (respuesta abierta -- calificada por IA, +5 XP)

En una frase: ¿tu ojo acerto en fuerza y direccion para `Percepción de
corrupción`, o te sorprendio algo del resultado?


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_ronda1()

#### Ronda 2 -- Antes de seguir, predice

Misma mecanica, tercera columna: **sin graficar todavia**, predice si
`Esperanza de vida saludable` vs. `Puntaje` te parece un patron fuerte o
debil, positivo o negativo.


In [ ]:
# 🔮 PREDICE (no se califica, es solo para ti)
mi_prediccion_ronda2 = "___"  # ej: "creo que es un patron fuerte y positivo"

#### ✅ Ejercicio 2 -- Grafica y calcula, otra vez (20 pts)

🔨 Mismo patron completo, ahora con `Esperanza de vida saludable` en el eje X
y `Puntaje` en el eje Y.

Variables que espera el autograder: `x_ex2`, `y_ex2`, `r_ex2`.


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex2()

In [ ]:
# ❓ Pregunta t7 -- ejecuta esta celda para verla y responder
grader.check_t7()

#### Ronda 3 -- PBI per cápita vs. Esperanza de vida saludable

Esta vez ninguna de las dos columnas es `Puntaje` -- las seis variables
economicas y sociales tambien pueden relacionarse **entre si**.


##### ✅ Ejercicio 3 -- Grafica y calcula (20 pts)

🔨 `PBI per cápita` (eje X) vs. `Esperanza de vida saludable` (eje Y).

Variables que espera el autograder: `x_ex3`, `y_ex3`, `r_ex3`.


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex3()

##### 💭 Reflexiona -- Ronda 3 (respuesta abierta -- calificada por IA, +5 XP)

`PBI per cápita` y `Esperanza de vida saludable` te deberia haber dado el r
mas alto que has visto hoy (mas alto incluso que cualquiera de los dos contra
`Puntaje`). ¿Por que crees que estas dos variables en particular se mueven
tan juntas?


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_ronda3()

#### Ronda 4 -- Apoyo social vs. Esperanza de vida saludable

Mismo patron, otro par sin `Puntaje`.


##### ✅ Ejercicio 4 -- Grafica y calcula (20 pts)

🔨 `Apoyo social` (eje X) vs. `Esperanza de vida saludable` (eje Y).

Variables que espera el autograder: `x_ex4`, `y_ex4`, `r_ex4`.


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex4()

#### Ronda 5 -- PBI per cápita vs. Apoyo social

`PBI per cápita` ya te dio el r más alto de la Ronda 3 (con `Esperanza de
vida saludable`). Repite el patron con otra variable distinta -- otra vez,
ninguna de las dos es `Puntaje`.


##### ✅ Ejercicio 5 -- Grafica y calcula (20 pts)

🔨 `PBI per cápita` (eje X) vs. `Apoyo social` (eje Y).

Variables que espera el autograder: `x_ex5`, `y_ex5`, `r_ex5`.


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex5()

##### 💭 Reflexiona -- Ronda 5 (respuesta abierta -- calificada por IA, +5 XP)

`PBI per cápita` te volvio a dar un r fuerte, esta vez con `Apoyo social`.
En 1-2 oraciones: ¿que tienen en comun estas dos relaciones fuertes que
calculaste hoy, y te parece razonable que el PBI se relacione
consistentemente fuerte con variables tan distintas?


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_pbi_apoyo()

#### Ronda 6 -- Generosidad vs. Percepción de corrupción

Hasta ahora, `Generosidad` casi no se relaciono con nada (r≈0.08 con
`Puntaje` en la Apertura). Prueba si pasa lo mismo con `Percepción de
corrupción`.


##### ✅ Ejercicio 6 -- Grafica y calcula (20 pts)

🔨 `Generosidad` (eje X) vs. `Percepción de corrupción` (eje Y).

Variables que espera el autograder: `x_ex6`, `y_ex6`, `r_ex6`.


In [ ]:
# 🔨 CONSTRUYE

# ============================
#      Tu codigo aqui
# ============================




In [ ]:
grader.check_ex6()

##### 💭 Reflexiona -- Ronda 6 (respuesta abierta -- calificada por IA, +5 XP)

`Generosidad` te dio un r cercano a 0 con casi todo lo que probaste hoy --
pero aqui ya no es tan chico. En 1-2 oraciones: ¿te parece razonable que
`Generosidad` casi no se relacione con nada mas, excepto con esta variable?
¿Que explicacion se te ocurre?


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_generosidad_corrupcion()

### 🧠 Chequeo de concepto

Ya calculaste `r` ocho veces hoy (Apertura + seis Rondas + el ejemplo
guiado). Antes de cerrar la clase, pon el concepto en tus propias palabras
-- sin usar ningun dataset ni numero especifico.


#### 💭 Reflexiona -- explica el concepto (respuesta abierta -- calificada por IA, +5 XP)

En 2-3 oraciones, sin usar ningun par de columnas como ejemplo: ¿que te dice
el coeficiente de correlacion, y que es lo que **nunca** te dice por si
solo?


In [ ]:
# 💭 Reflexiona -- ejecuta esta celda para responder
grader.check_reflexion_concepto()

---
## 🧭 Quiz de Cierre

Ya calculaste seis correlaciones reales hoy. Antes de cerrar: una pregunta
para recordar lo que viste en esta clase, y tres preguntas con ejemplos
**nuevos** -- para probar si el concepto de correlacion se te quedo mas
alla del dataset de felicidad.


In [ ]:
# ❓ Pregunta t8 -- ejecuta esta celda para verla y responder
grader.check_t8()

In [ ]:
# ❓ Pregunta t9 -- ejecuta esta celda para verla y responder
grader.check_t9()

In [ ]:
# ❓ Pregunta t10 -- ejecuta esta celda para verla y responder
grader.check_t10()

In [ ]:
# ❓ Pregunta t11 -- ejecuta esta celda para verla y responder
grader.check_t11()

In [ ]:
# ✅ CHECKPOINT -- necesitas 80% en esta seccion para continuar
grader.check_mini_a()

---
## 🏁 Fin de la Clase 1 -- Semana 3

Aprendiste a leer un patron con el ojo, a ponerle un numero exacto en el
mismo paso, y a desconfiar de ese numero cuando corresponde. La **Semana 4**
continua la Mision 2: vas a descubrir que el patron general puede esconder --
o hasta invertir -- lo que pasa dentro de cada grupo, y vas a elegir tu propio
par de variables para un mini-proyecto.


In [ ]:
grader.resumen()